# Ⅰ第1回 演習1（E1-1）環境ベンチマーク

**今日の問い**：良いAIとは何を測ることか

この演習では自分のマシンの「機種・速度・メモリ」を測り，グループの台帳（CSV）に投稿する．
ブロックを上から順に実行し，`TODO` の箇所を埋める．実行時間の目安は 3 分以内．

**この演習で身につけること**
- GPU（MPS）は CPU と別に裏で計算する．GPU の計算の終わりを待つ（同期する）までは，正しい時間は測れない
- 「速い」には 2 種類ある：メモリの読み書きで速さが決まる処理と，計算の量で速さが決まる処理．どちらになるかは処理で変わる


## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) コードテンプレート①：ライブラリのインポート

教科書 2.3 節のテンプレート①に，授業用の計測モジュール（`common`）を加えたもの．

In [ ]:
import time                       # 時間計測
import json
import numpy as np
import torch                      # PyTorch
import pandas as pd
from common.device import get_device, synchronize, machine_info   # MPS/CPU 選択と同期
from common import memory, bench                                   # ピークメモリ計測・ベンチマーク
from common.logger import ResultLogger                             # 台帳（CSV）への記録
from common.todo import todo_check                                 # TODO の書き忘れを見つける
print("torch", torch.__version__)

## (2) 計測：機種情報の取得

チップ名・メモリ量・OS・PyTorch の版を取る．この行がそのまま台帳の1行になる．

In [ ]:
info = machine_info()
for k, v in info.items():
    print(f"{k:14s}: {v}")

## (3) コードテンプレート⑤：使用デバイスの選択

教科書 2.3 節のテンプレート⑤と同じ 3 分岐（cuda → mps → cpu）．受講者の MacBook Air では MPS（Apple GPU）が選ばれる．
授業用の `get_device()` は⑤に，16GB 機の実使用可能量に合わせた **11GB のメモリ上限** を加えたもの．

In [ ]:
# コードテンプレート⑤
# TODO 1: elif の後ろの「...」を消して，MPS（Apple の GPU）が使えるかを調べる式を書く
#   ヒント：1 行上の torch.cuda.is_available() は「NVIDIA の GPU が使えるか」を調べる式．MPS 版も同じ形
#   （「...」は「ここに書く」という目印．消さずに実行すると todo_check の行で止まる）
if torch.cuda.is_available():
    device = torch.device("cuda")
elif ...:
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用デバイス: {device}")
todo_check("TODO 1")       # 「...」が残っていたら止める

device = get_device()      # 授業では以降これを使う（⑤ ＋ メモリ上限）
print("使用デバイス:", device, "| メモリ上限:", memory.cap_torch())

## (4) 計測：同期しないと何が起きるか

Python（CPU）は GPU に計算を頼むとすぐ次の行へ進み，GPU は裏で計算を続ける．
**同期** とは，CPU が GPU の計算の終わりを待って足並みをそろえること（`synchronize(device)`）．

- 同期なし：待たずに時計を読む → 「頼んだだけ」の時間
- 同期あり：GPU の計算が終わるのを待ってから時計を読む → 本当の計算時間
- 同期比 ＝ 同期ありの時間 ÷ 同期なしの時間

**型：測りたい処理を synchronize で前後から挟む**

```python
synchronize(device)            # ① 前に頼んだ計算を終わらせる
t0 = time.perf_counter()       # ② 時計スタート
（測りたい処理）                # ③
synchronize(device)            # ④ ③の計算が全部終わるまで待つ
t = time.perf_counter() - t0   # ⑤ 時計ストップ
```

In [ ]:
N = 2048
a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)
_ = a @ b                                    # ウォームアップ（初回はカーネルのコンパイルで遅い）

# --- 同期なし：GPU の計算の終わりを待たずに時計を読む ---
t0 = time.perf_counter()
for _ in range(10):
    c = a @ b
t_nosync = (time.perf_counter() - t0) / 10

# --- 同期あり：GPU の計算が終わるのを待ってから時計を読む ---
# TODO 2: 測りたい処理（下の for 文）を synchronize(device) で前後から挟む．
#   時計スタートの前に 1 行，for 文の後（時計ストップの前）に 1 行．足さなくてもエラーは出ない．下の判定で確かめる
t0 = time.perf_counter()
for _ in range(10):
    c = a @ b
t_sync = (time.perf_counter() - t0) / 10

print(f"同期なし（待たない）: {t_nosync*1000:8.3f} ms / 回")
print(f"同期あり（待つ）  : {t_sync*1000:8.3f} ms / 回")
print(f"同期比（あり÷なし）: {t_sync / max(t_nosync, 1e-9):.1f} 倍   ← MPS では大きい．CPU ではほぼ 1")

# 判定：2048×2048 の行列積は約 170 億回の演算．Mac で最速級の GPU でも 1 回 1 ms 以上かかる
gflops_sync = 2 * N ** 3 / t_sync / 1e9
if device.type == "mps" and t_sync < 0.5e-3:
    print(f"NG：同期ありが {t_sync*1000:.3f} ms は速すぎる（{gflops_sync:,.0f} GFLOPS 相当）．synchronize(device) が入っていない")
elif device.type == "mps":
    print(f"OK：同期ありは 1 回 {t_sync*1000:.1f} ms（{gflops_sync:,.0f} GFLOPS 相当）．GPU の計算を待てている")
else:
    print("CPU で実行中：CPU は待つ必要がないので比はほぼ 1 になる")

## (5) 計測 B1：実効メモリ帯域（GB/s）

大きなベクトルの加算 `c = a + b` は計算そのものはすぐ終わり，**メモリからデータを読み書きする時間** で全体の速さが決まる（ボトルネックになる）．
**考え方（TODO 3）**：1要素で 読み2個＋書き1個 ＝ 3個 → 1個は4バイト（float32）→ 要素は n 個．この3つを掛ける．
図とアニメーションは授業ページ演習1「動くバイト数とは」．

**もう 1 つの落とし穴**：GPU は最初の実行に準備の時間が入るので，最初の数回は捨てる（ウォームアップ）．
1 回だけの値はぶれるので，**2 秒ほど回して中央値** を取る．

In [ ]:
def timed_loop(fn, min_sec=2.0, min_iters=5):
    """fn を最低 min_sec 秒回し，1 回あたりの時間の中央値を返す"""
    for _ in range(2):
        fn()                                  # ウォームアップ
    synchronize(device)
    times = []
    t_start = time.perf_counter()
    while len(times) < min_iters or time.perf_counter() - t_start < min_sec:
        t0 = time.perf_counter()
        fn()
        synchronize(device)                   # 計算の完了を待ってから時計を読む
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

n = 64 * 1024 * 1024                          # 6400万要素（256MB × 3）
a = torch.ones(n, device=device)
b = torch.ones(n, device=device)
sec = timed_loop(lambda: torch.add(a, b))

# TODO 3: 下の「...」を消して，1回の加算で動くバイト数を式で書く（読み2本 + 書き1本，float32 = 4 バイト）
bytes_moved = ...
todo_check("TODO 3")                          # 「...」が残っていたら止める
bandwidth_gbps = bytes_moved / sec / 1e9
print(f"B1 帯域: {bandwidth_gbps:.1f} GB/s  ({sec*1000:.2f} ms/回，中央値)")
nominal = bench.nominal_bandwidth_gbps(info["chip"])
if nominal:
    print(f"   公称値 {nominal:g} GB/s（{info['chip']}）の {bandwidth_gbps / nominal * 100:.0f}%   ← 公称値付近までが目安．大きく超えたら（120% 以上など）式（TODO 3）を疑う")
else:
    print("   この機種の公称値は表に無い（授業ページの表を見る）")

## (6) 計測 B2：行列積の実効演算性能（float32，GFLOPS）

行列積は読み書きより **計算の量** で全体の速さが決まる．

**考え方（TODO 4）**：C の1マスに 掛け算 N 回＋足し算 N−1 回 → C のマスは N×N 個．厳密には 2N³ − N² 回だが，性能測定では慣例として **2N³** と数える（N³ は `N ** 3`）．
図とアニメーションは授業ページ演習1「行列積の演算数とは」．

In [ ]:
N = 2048
a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)
sec = timed_loop(lambda: a @ b)

# TODO 4: 下の「...」を消して，N×N 行列積 1 回の演算数（乗算と加算の回数）を式で書く
flops = ...
todo_check("TODO 4")                          # 「...」が残っていたら止める
gflops = flops / sec / 1e9
print(f"B2 演算: {gflops:.0f} GFLOPS  ({sec*1000:.2f} ms/回，中央値)")

## (7) 計測 B3：学習ステップ時間（ms/step）

授業で使う小型 Transformer（A3，約30万パラメータ）の 1 ステップ（順伝播＋逆伝播＋更新）にかかる時間．
これが授業中の「待ち時間」を決める．

In [ ]:
step_ms, n_params = bench.train_step_ms(device)
print(f"B3 学習: {step_ms:.2f} ms/step  (モデル {n_params:,} パラメータ, バッチ 128)")

## (8) 計測：メモリ

1GB のテンソルを確保したときのピークを測る．`PeakSampler` はブロック内の最大使用量を裏で採取する．

In [ ]:
with memory.PeakSampler("driver") as ps:             # MPS ドライバが確保した量（CPU 時は RSS）
    big = torch.empty(256 * 1024 * 1024, device=device)  # 1GB
    big.fill_(1.0)
    synchronize(device)
del big
print(f"ピーク: {ps.peak_mb:.0f} MB  増分: {ps.delta_mb:.0f} MB  種別: {ps.kind}")
print("この機の搭載メモリ:", info["memory_gb"], "GB")

## (9) 記録：台帳へ投稿

測った値を共通形式の CSV に追記する．グループの全員が投稿し，`aggregate/c1d1_env.py` で提出された分を1枚にまとめる．

In [ ]:
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d1", exercise="ex1", device=device)
logger.log_many({
    "bandwidth_gbps": bandwidth_gbps,
    "matmul_gflops": gflops,
    "train_step_ms": step_ms,
    "memory_gb": info["memory_gb"],
    "alloc_1gb_peak_mb": ps.peak_mb,
    "sync_ratio": t_sync / max(t_nosync, 1e-9),
}, condition=info["chip"] or "unknown")
print("書き込み先:", logger.path)
# 台帳に貼る 1 行
print(f"{logger.machine} | {device} | B1 {bandwidth_gbps:.0f} GB/s | B2 {gflops:.0f} GFLOPS | B3 {step_ms:.1f} ms/step")

## (10) 記録の確認

自分が書いた CSV を読み戻す．この形式が全14回で共通になる．

In [ ]:
df = pd.read_csv(logger.path)
df.tail(6)

## (11) グループディスカッション（5 分）→ グループ内プレゼン1（1 人 2 分）

presenter が進行し，グループの 4 台の数値を並べて **全員が 1 回は発言** する．recorder は結論をまとめる．

**討議の問い**（順に）
1. 同期あり／なしの比はグループ内で何倍から何倍まであったか．なぜ機種で違うか
2. 帯域は公称値の何 % 出たか
3. B1（帯域）と B2（演算）で，グループ内の機種差が大きいのはどちらか．その理由は

**プレゼン1 の型**：presenter がグループの結論を 2 分 → implementer・verifier・recorder が 1 分ずつ補足

In [ ]:
# 討議用にグループの 4 台ぶんを並べる（各自の CSV が同じフォルダにある場合）．無い場合は自分の 1 行だけ出る．
df_all = pd.read_csv(logger.path)
latest = df_all.sort_values("timestamp").groupby(["machine", "metric_name"]).tail(1)
board = latest.pivot_table(index="machine", columns="metric_name", values="metric_value")
cols = [c for c in ["sync_ratio", "bandwidth_gbps", "matmul_gflops", "train_step_ms", "memory_gb"] if c in board]
display(board[cols].round(1))